# Bangla Fake News Detection

## Objective
This project is edutational purpose, trained with "BanFakeNews", famous Kaggle Data(50K). 

## Import Necessary libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Import Datasets

In [2]:
auth= pd.read_csv("data/Authentic-48K.csv")
fake= pd.read_csv("data/Fake-1K.csv")

In [3]:
# Check columns
auth.columns.tolist()==fake.columns.tolist()

True

In [4]:
# Concate Both files
news=pd.concat([auth, fake], ignore_index=True)
news

,articleID,domain,date,category,headline,content,label
0,1,jagonews24.com,2018-09-19 17:48:18,Education,"হট্টগোল করায় বাকৃবিতে দুইজন বরখাস্ত, ৬ জনকে শোকজ",গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে ...,1
1,2,jagonews24.com,2018-09-19 17:48:19,National,মালয়েশিয়ায় কর্মী পাঠানোর ব্যবস্থা নেয়ার সুপারিশ,বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রম...,1
2,3,jagonews24.com,2018-09-19 17:48:20,National,প্রেমের প্রস্তাবে রাজি না হওয়ায় স্কুলছাত্রীকে ...,নরসিংদীর মনোহরদীতে প্রেমের প্রস্তাবে রাজি না হ...,1
3,4,jagonews24.com,2018-09-19 17:48:21,Crime,মেডিয়েশনই মামলাজট নিরসনের পথ : বিচারপতি আহমেদ ...,সুপ্রিম কোর্টের হাইকোর্ট বিভাগের বিচারপতি আহমে...,1
4,5,jagonews24.com,2018-09-19 17:48:21,National,টকশোতে বক্তব্য দিতে গিয়ে জাপা নেতার মৃত্যু,মাদারীপুর সদরের উপজেলার লেকেরপাড়ে একটি বেসরকার...,1
...,...,...,...,...,...,...,...
49972,1303,bengalbeats.com,2019-03-31T21:34:42+00:00,Miscellaneous,রাজশাহীতে স্বামীর হাঁচির আওয়াজে ঘুমের মধ্যেই স...,রাজশাহীর সাহেব বাজার মোড়ে রাতে ঘুমের মধ্যে হাস...,0
49973,1304,bengaliviralnews.com,2018-01-16T15:36:12+00:00,Miscellaneous,IAS ইন্টারভিউ এ মেয়েটিকে জিজ্ঞসা করা হল মেয়েদে...,IAS পরীক্ষায় এমন এমন অবাক করে দেওয়া প্রশ্ন এসে...,0
49974,1305,channeldhaka.news,2018-12-03T18:09:08+00:00,Crime,শাওমি মোবাইল বিস্ফোরণে কলেজছাত্রের মৃত্যু!,শাওমি মোবাইল ফোন বিস্ফোরণ হয়ে স্বপ্নীল মজুমদা...,0
49975,1306,earki.com,"১৭:৩৬, জানুয়ারি ০৭, ২০১৯",Miscellaneous,ফেসবুকে যেভাবে নতুন মন্ত্রীদের অভিনন্দন বার্তা...,আপনাকে নতুন মন্ত্রিসভার সদস্যদের অভিনন্দন জানা...,0


# Class Imbalance Check

In [5]:
news["label"].value_counts()

label
1    48678
0     1299
Name: count, dtype: int64

In [6]:
news["label"].value_counts(normalize=True)*100

label
1    97.400804
0     2.599196
Name: proportion, dtype: float64

## Confirmed — 97.4% authentic vs 2.6% fake, so imbalance-handling isn't optional here; a model that just predicts "authentic" every time would already hit 97.4% accuracy while being useless.

## Next step: before you touch any imbalance-handling technique (SMOTE, class weights, etc.), you need to split into train/test first — and do it in a way that preserves this ratio in both sets.

# Train-Test-Split

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X=news["content"]
y=news["label"]
X_train, X_test, y_train, y_test=train_test_split(X,y , test_size=0.2, stratify=y, random_state=42)

In [9]:
#Check class imbalance after stratification
y_train.value_counts(normalize=True)*100
y_test.value_counts(normalize=True)*100

label
1    97.39896
0     2.60104
Name: proportion, dtype: float64

## So the class imbalance is preserved after train_test_split

# Handle Class Imbalance

### A few real options, each with different tradeoffs — worth understanding before picking one blindly:

    Class weighting — tell the NB model to weight the minority class more heavily during training. Simplest option, no synthetic data involved. 
    Random oversampling — duplicate minority-class (fake) examples until balanced. Easy, but risks overfitting to those exact duplicated examples.
    SMOTE — synthetically generates new minority examples by interpolating between existing ones in feature space. Standard for tabular data, but there's a subtlety for text: SMOTE was designed for continuous numeric features, and applying it directly to sparse TF-IDF/count vectors is common in practice but statistically shakier than on structured data. Worth knowing this caveat exists even if you still use it.
    Undersampling the majority class — throw away most of the 48K authentic examples to match the ~1K fake count. Simple, but wastes a lot of data — probably not ideal here given how few fake examples you already have.
    Just report the right metrics and skip resampling entirely — for Naive Bayes specifically, since it's a generative model working off priors and likelihoods, an interesting angle is to manually adjust priors rather than resample data at all (NB explicitly separates the prior P(class) from the likelihood, so you can override this cleanly). This ties nicely into "understanding the math" rather than treating resampling as a black-box fix.

    Given this is a from-scratch NB implementation (per our earlier plan), option 5 is actually the most interesting choice for your specific project — you can show you understand why priors matter and manually control them, which most portfolio projects using imblearn.SMOTE() blindly don't demonstrate.

# Base-line Model

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.33      0.72      0.45       260
           1       0.99      0.96      0.98      9736

    accuracy                           0.95      9996
   macro avg       0.66      0.84      0.71      9996
weighted avg       0.98      0.95      0.96      9996



In [11]:
# Check, if the tokens are meaningfull or not 
vectorizer.get_feature_names_out()[:20]

array(['000', '0054497', '01', '01557229215', '01558136814', '01711',
       '017186959072', '01723236946', '01797170309', '01912990597',
       '01947232734', '01955803728', '0200003244054', '03', '06', '09',
       '0px', '0১', '0২', '10'], dtype=object)

### That confirms it — this is exactly the "garbage tokens" problem I flagged. Your vocabulary right now is dominated by phone numbers, IDs, and stray digits/HTML artifacts (0px is a CSS leftover, probably from scraped web content that wasn't cleaned). None of this is meaningful signal for classifying fake vs authentic news — it's noise that'll bloat your vocabulary and dilute the real word-based patterns NB needs to work with.

    This needs to be fixed before your baseline numbers mean anything. A "97% accuracy" baseline built on phone-number tokens isn't a real baseline — it's noise you'd have to redo anyway.

### Two separate problems mixed together here:
### Numeric noise (phone numbers, IDs, plain digits) — should almost certainly be stripped entirely for this task. News articles don't need digit-strings as classification features (unless you have a specific hypothesis that fake news articles cite phone numbers more often — possible, but treat that as a deliberate feature, not an accident).
### Leftover markup (0px) — signals the raw content text has unstripped HTML/CSS remnants from scraping. This needs proper cleaning, not just tokenization tweaks.

# Clean Text

In [12]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Remove HTML tags / leftover markup
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)
    
    # Remove phone-number-like digit sequences (long digit runs, Bangla + Latin digits)
    text = re.sub(r'[0-9০-৯]{4,}', ' ', text)
    
    # Remove any remaining standalone digits/numbers
    text = re.sub(r'[0-9০-৯]+', ' ', text)
    
    # Remove CSS/artifact fragments like "0px", "10em" etc. (in case any survive)
    text = re.sub(r'\b\w*px\b|\b\w*em\b', ' ', text)
    
    # Keep only Bangla characters (Unicode range) and basic whitespace
    # Bangla Unicode block: U+0980–U+09FF
    text = re.sub(r'[^\u0980-\u09FF\s]', ' ', text)
    
    # Collapse multiple spaces into one, strip leading/trailing
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Check cleaned text

In [13]:
news['content_clean'] = news['content'].apply(clean_text)

# check a few before/after
for i in range(3):
    print("BEFORE:", news['content'].iloc[i][:200])
    print("AFTER :", news['content_clean'].iloc[i][:200])
    print()



BEFORE: গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে (বাকৃবি) উপাচার্যের কার্যালয়ে হট্টগোলের ঘটনায় দুইজনকে সাময়িক বরখাস্ত ও ছয় জনকে শোকজ করেছে বিশ্ববিদ্যালয় প্রশাসন। বুধবার বিশ্ববিদ্যালয় বাকৃবি রেজিস্ট্রার স
AFTER : গত সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে বাকৃবি উপাচার্যের কার্যালয়ে হট্টগোলের ঘটনায় দুইজনকে সাময়িক বরখাস্ত ও ছয় জনকে শোকজ করেছে বিশ্ববিদ্যালয় প্রশাসন বুধবার বিশ্ববিদ্যালয় বাকৃবি রেজিস্ট্রার সাইফুল 

BEFORE: বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রমিক পাঠানোর প্রয়োজনীয় সব ধরনের ব্যবস্থা নেয়ার সুপারিশ করেছে সংসদীয় কমিটি। ২৫ সেপ্টেম্বর প্রবাসী কল্যাণ ও বৈদেশিক কর্মসংস্থান মন্ত্রণালয়ের মন্ত্রীর নেতৃত্বে
AFTER : বাংলাদেশের বৃহৎ শ্রমবাজার মালয়েশিয়ায় আবার শ্রমিক পাঠানোর প্রয়োজনীয় সব ধরনের ব্যবস্থা নেয়ার সুপারিশ করেছে সংসদীয় কমিটি সেপ্টেম্বর প্রবাসী কল্যাণ ও বৈদেশিক কর্মসংস্থান মন্ত্রণালয়ের মন্ত্রীর নেতৃত্বে মাল

BEFORE: নরসিংদীর মনোহরদীতে প্রেমের প্রস্তাবে রাজি না হওয়ায় তানজিমা (১৪) নামে এক স্কুলছাত্রীকে কুপিয়ে জখম করেছে মফিজুল ইসলাম (১৮) নামে এক যুবক। বুধবার সকালে উপজেলা

In [15]:
# check for empty strings after cleaning (bad sign if many)
print((news['content_clean'].str.strip() == '').sum())

6


# Only 6 rows become empty. 6 out of 49k is negligble, we can remove these

In [17]:
news = news[news['content_clean'].str.strip() != ''].reset_index(drop=True)

# Retrain Model

In [18]:
X = news['content_clean']
y = news['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# confirm balance held
print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

label
1    97.400941
0     2.599059
Name: proportion, dtype: float64
label
1    97.398699
0     2.601301
Name: proportion, dtype: float64


# Vectorize Again

In [20]:
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(vectorizer.get_feature_names_out()[:20])

['অঅক' 'অঅরও' 'অঅল' 'অঅহমদ' 'অই' 'অইছ' 'অইট' 'অইত' 'অইব' 'অইল' 'অইয়' 'অউট'
 'অউর' 'অউস' 'অও' 'অওত' 'অওরঙ' 'অওয়' 'অক' 'অকট']
